Expects the vfi-golden-set + model dataset as input

In [ ]:
import os
import cv2
import shutil
import numpy as np
import tensorflow as tf
from tensorflow import keras

# --- CONFIGURATION ---
MODEL_PATH = "/kaggle/input/vfi-golden-set-model/golden_set_septuplets/models/vfi_septuplet_epoch_35.keras"
INPUT_DATA_DIR = "/kaggle/input/vfi-golden-set-model/golden_set_septuplets/sequences"
OUTPUT_ROOT = "/kaggle/working/golden_set"

MODEL_INPUT_SIZE = (256, 256)
WEBP_QUALITY = 85 

# --- HELPER FUNCTIONS ---

def get_image_info(path):
    img = cv2.imread(path)
    if img is None: return None
    h, w, _ = img.shape
    return (h, w)

def load_and_preprocess(path):
    img = cv2.imread(path)
    if img is None: return None
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img_small = cv2.resize(img, MODEL_INPUT_SIZE, interpolation=cv2.INTER_AREA)
    return img_small.astype('float32') / 255.0

def save_optimized_webp(img_array, path, target_dims=None):
    """Saves float32 array as optimized WebP, upscaling if needed."""
    img_uint8 = (img_array * 255.0).clip(0, 255).astype('uint8')
    if target_dims:
        h_orig, w_orig = target_dims
        img_uint8 = cv2.resize(img_uint8, (w_orig, h_orig), interpolation=cv2.INTER_CUBIC)
    
    cv2.imwrite(path, cv2.cvtColor(img_uint8, cv2.COLOR_RGB2BGR), [int(cv2.IMWRITE_WEBP_QUALITY), WEBP_QUALITY])

def process_sequence(model, sequence_folder):
    seq_id = os.path.basename(sequence_folder)
    
    # Verify existence of im1.webp
    test_frame = os.path.join(sequence_folder, "im1.webp")
    if not os.path.exists(test_frame):
        return False

    # Define paths
    interp_folder = os.path.join(OUTPUT_ROOT, "interpolation", seq_id)
    pred_folder = os.path.join(OUTPUT_ROOT, "prediction", seq_id)
    os.makedirs(interp_folder, exist_ok=True)
    os.makedirs(pred_folder, exist_ok=True)

    orig_dims = get_image_info(test_frame)
    if not orig_dims: return False

    # Load available frames
    frames_small = []
    missing_im7 = False
    
    for i in range(1, 8):
        f_path = os.path.join(sequence_folder, f"im{i}.webp")
        if not os.path.exists(f_path):
            if i == 7:
                print(f"⚠️ Sequence {seq_id} is missing im7.webp. Using prediction as placeholder ground truth.")
                missing_im7 = True
                # Add a dummy frame for now to keep the list index correct
                frames_small.append(np.zeros((*MODEL_INPUT_SIZE, 3), dtype='float32'))
                continue
            else:
                return False # We still strictly need 1-6 for interpolation/prediction
        
        img = load_and_preprocess(f_path)
        if img is None: return False
        frames_small.append(img)

    # --- 1. INTERPOLATION (Predicting im4) ---
    # Input: 1,2,3,5,6,7
    # Note: If im7 is missing, interpolation will use the dummy zero-frame for im7.
    # This will look weird, but maintains the pipeline.
    interp_input = np.concatenate([frames_small[0], frames_small[1], frames_small[2], 
                                   frames_small[4], frames_small[5], frames_small[6]], axis=-1)
    interp_input = np.expand_dims(interp_input, axis=0)
    interp_results = model.predict(interp_input, verbose=0)
    im4_pred = interp_results[1][0] 

    # Copy frames to interpolation folder
    for i in range(1, 8):
        src = os.path.join(sequence_folder, f"im{i}.webp")
        dest = os.path.join(interp_folder, f"im{i}.webp")
        if os.path.exists(src):
            shutil.copy2(src, dest)
        elif i == 7:
            # For the interpolation folder, we'll use im7_pred (generated below) as a fallback im7
            pass 

    save_optimized_webp(im4_pred, os.path.join(interp_folder, "im4_pred.webp"), orig_dims)

    # --- 2. PREDICTION (Predicting im7) ---
    # Input: 1,2,3,4,5,6
    pred_input = np.concatenate(frames_small[0:6], axis=-1)
    pred_input = np.expand_dims(pred_input, axis=0)
    pred_results = model.predict(pred_input, verbose=0)
    im7_pred = pred_results[0][0] 

    # Copy frames to prediction folder
    for i in range(1, 8):
        src = os.path.join(sequence_folder, f"im{i}.webp")
        dest = os.path.join(pred_folder, f"im{i}.webp")
        if os.path.exists(src):
            shutil.copy2(src, dest)
        elif i == 7 and missing_im7:
            # SPECIAL CASE: Save the prediction as the 'Ground Truth' im7.webp 
            # so the folder isn't broken.
            save_optimized_webp(im7_pred, dest, orig_dims)
            # Also save it to the interpolation folder for consistency
            save_optimized_webp(im7_pred, os.path.join(interp_folder, "im7.webp"), orig_dims)

    save_optimized_webp(im7_pred, os.path.join(pred_folder, "im7_pred.webp"), orig_dims)
    
    return True

def main():
    if os.path.exists(OUTPUT_ROOT): shutil.rmtree(OUTPUT_ROOT)
    
    print(f"🔍 Searching for sequences in: {INPUT_DATA_DIR}")
    
    try:
        model = keras.models.load_model(MODEL_PATH, compile=False)
    except Exception as e:
        print(f"❌ ERROR loading model: {e}"); return

    # Improved recursive search for folders containing im1.webp
    valid_sequences = []
    for root, dirs, files in os.walk(INPUT_DATA_DIR):
        if "im1.webp" in files:
            valid_sequences.append(root)
    
    valid_sequences.sort()
    total_found = len(valid_sequences)
    print(f"✅ Found {total_found} valid sequences.")

    if total_found == 0:
        print("❌ No sequences found! Check if im1.webp exists in your dataset.")
        return

    processed_count = 0
    for seq in valid_sequences:
        if process_sequence(model, seq):
            processed_count += 1
            if processed_count % 10 == 0:
                print(f"🚀 Processed {processed_count}/{total_found}...")

    # Final Zipping
    zip_name = '/kaggle/working/vfi_golden_set_webp'
    if os.path.exists(OUTPUT_ROOT) and os.listdir(OUTPUT_ROOT):
        print(f"\n🤐 Zipping {processed_count} results...")
        shutil.make_archive(zip_name, 'zip', OUTPUT_ROOT)
        print(f"✨ Success! Archive ready.")
    else:
        print("\n❌ Output empty.")

if __name__ == "__main__":
    main()

🔍 Searching for sequences in: /kaggle/input/vfi-golden-set-model/golden_set_septuplets/sequences
✅ Found 110 valid sequences.
🚀 Processed 10/110...
🚀 Processed 20/110...
🚀 Processed 30/110...
🚀 Processed 40/110...
⚠️ Sequence 046 is missing im7.webp. Using prediction as placeholder ground truth.
🚀 Processed 50/110...
🚀 Processed 60/110...
🚀 Processed 70/110...
🚀 Processed 80/110...
